# K=64 pairlab60, 60 -> 120 epochs, two cards: seeds 0, 1

Resumes results_k64/pairlab60 (beams + channels 8 + --keep-best --pair-lab, 60 epochs) from its
epoch-60 checkpoints, mounted from that kernel's output, to 120 epochs, one
seed per T4. The cosine
schedule is re-anchored on the 120-epoch horizon (learning rate restarts at
1.5e-3). T4 with torch 2.11.0+cu128 pinned and asserted.

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0",
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import os, subprocess, sys

def launch(args, gpu, log):
    """Start a repo module on one card; its output goes to `log`."""
    # unbuffered, with faulthandler: a killed or crashed child still leaves
    # its last lines and a traceback in `log`
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONFAULTHANDLER="1")
    return subprocess.Popen([sys.executable, "-m"] + args, cwd="/kaggle/working/repo",
                            env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def tail(log, n=3000):
    return open(log).read()[-n:]

cards = [l for l in subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                   capture_output=True, text=True).stdout.splitlines() if l.strip()]
NGPU = len(cards)
print("cards:", cards)

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
TAG, SEEDS, EPOCHS, MAX_SECONDS = 'pairlab120', [0, 1], 120, 39000
OUT = "/kaggle/working/results_k64/" + TAG
CKPT = "/kaggle/working/checkpoints/" + TAG
LOGS = "/kaggle/working/logs/" + TAG
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '64', '--eval-chunk-size', '1024', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile', '--keep-best', '--pair-lab']
BASE[BASE.index("DATA")] = DATA
for d in (OUT, LOGS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
pathlib.Path(OUT, "bundle_sha256.txt").write_text('56b164e90d91' + chr(10))
print("data:", DATA, "| bundle", '56b164e90d91')

In [ ]:
# On-card guard: the forward pass must be the published function on this
# GPU too, in float64. "1 passed" is required; a skip means no CUDA.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1500:], g.stderr[-1500:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
# The epoch-60 checkpoint of seed 0, from the mounted output of nsanya/so3c-k64-pairlab60-seeds01: check it
# against the committed copy's sha256 and copy it where --resume looks (the
# experiment and model in the file name are the run's).
import glob, hashlib, pathlib, torch
found = glob.glob("/kaggle/input/**/checkpoints/pairlab60/seed0/*seed0.pt", recursive=True)
assert len(found) == 1, "expected one mounted seed-0 checkpoint, found %r" % found
raw = pathlib.Path(found[0]).read_bytes()
assert hashlib.sha256(raw).hexdigest() == 'df098ad4988cebf751c771807bef544b5176a98893afd1d267a83d7bcb94c720', "mounted checkpoint differs from the committed one"
ck = pathlib.Path(CKPT) / "seed0" / "top_tagging_canonical__so3c_message_set__seed0.pt"
ck.parent.mkdir(parents=True, exist_ok=True)
ck.write_bytes(raw)
state = torch.load(ck, map_location="cpu", weights_only=False)
assert state["epoch"] == 60 and len(state["history"]) == 60, (state["epoch"], len(state["history"]))
assert abs(state["history"][-1]["val_loss"] - 0.16073009371757507) < 1e-9, state["history"][-1]["val_loss"]
print("seed 0 checkpoint ok: epoch 60, val loss %.5f" % state["history"][-1]["val_loss"])
del state, raw

In [ ]:
# The epoch-60 checkpoint of seed 1, from the mounted output of nsanya/so3c-k64-pairlab60-seeds01: check it
# against the committed copy's sha256 and copy it where --resume looks (the
# experiment and model in the file name are the run's).
import glob, hashlib, pathlib, torch
found = glob.glob("/kaggle/input/**/checkpoints/pairlab60/seed1/*seed1.pt", recursive=True)
assert len(found) == 1, "expected one mounted seed-1 checkpoint, found %r" % found
raw = pathlib.Path(found[0]).read_bytes()
assert hashlib.sha256(raw).hexdigest() == '2b000be485a8cb15237ab7c6e11f77e4697fce60c1e5b33ff1576f889ee15af8', "mounted checkpoint differs from the committed one"
ck = pathlib.Path(CKPT) / "seed1" / "top_tagging_canonical__so3c_message_set__seed1.pt"
ck.parent.mkdir(parents=True, exist_ok=True)
ck.write_bytes(raw)
state = torch.load(ck, map_location="cpu", weights_only=False)
assert state["epoch"] == 60 and len(state["history"]) == 60, (state["epoch"], len(state["history"]))
assert abs(state["history"][-1]["val_loss"] - 0.15742020308971405) < 1e-9, state["history"][-1]["val_loss"]
print("seed 1 checkpoint ok: epoch 60, val loss %.5f" % state["history"][-1]["val_loss"])
del state, raw

In [ ]:
# Smoke on both cards at once (fresh models, separate directories): one epoch
# on 20k jets per card. Each must write a result.
import glob, json
smoke = []
for i in range(min(NGPU, len(SEEDS))):
    d = "/kaggle/working/smoke/%s/card%d" % (TAG, i)
    smoke.append((d, launch(["benchmarks.run_top_tagging"] + BASE +
                            ["--epochs", "1", "--max-train-samples", "20000",
                             "--seed", "0", "--results-dir", d],
                            i, LOGS + "/smoke_card%d.log" % i)))
codes = [p.wait() for _, p in smoke]
works = []
for i, ((d, _), rc) in enumerate(zip(smoke, codes)):
    found = glob.glob(d + "/*.json")
    print("card %d: exit %d, result %s" % (i, rc, bool(found)))
    if found:
        works.append(i)
    else:
        print(tail(LOGS + "/smoke_card%d.log" % i))
assert works, "the smoke run failed on every card"
PARALLEL = len(works) >= len(SEEDS)
print("parallel:" if PARALLEL else "only card(s) %s work: seed %d alone" % (works, SEEDS[0]),
      "seeds", SEEDS if PARALLEL else SEEDS[:1])

In [ ]:
import glob, time
todo = SEEDS if PARALLEL else SEEDS[:1]
procs = []
for i, s in enumerate(todo):
    gpu = i if PARALLEL else works[0]
    procs.append((s, launch(["benchmarks.run_top_tagging"] + BASE +
                            ["--epochs", str(EPOCHS), "--seed", str(s),
                             "--results-dir", OUT + "/seed%d" % s,
                             "--ckpt-dir", CKPT + "/seed%d" % s,
                             "--resume", "--max-seconds", str(MAX_SECONDS)],
                            gpu, LOGS + "/seed%d.log" % s)))
    print("seed %d on card %d" % (s, gpu), flush=True)
BAD = []
for s, p in procs:
    rc = p.wait()
    print("=== seed %d: exit %d ===" % (s, rc))
    print(tail(LOGS + "/seed%d.log" % s, 2000))
    if rc != 0 or not glob.glob(OUT + "/seed%d/*.json" % s):
        BAD.append(s)
print("failed seeds:", BAD or "none")
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
# Each continued seed against its own epoch 60 (AUC, rejection at 30%, final
# validation loss), and against the 120-epoch headline.
import json, glob
OWN60 = {0: (0.16073009371757507, 0.9841706950893652, 1346.0933333333332), 1: (0.15742020308971405, 0.9846654727988882, 1246.3827160493827)}
REF120 = (0.98392, 1201.9, 0.16184)
for f in sorted(glob.glob(OUT + "/seed*/*.json")):
    r = json.load(open(f)); t = r["test_metrics"]; env = r["env"]
    h = r["history"]; vl = [e["val_loss"] for e in h]; s = r["seed"]
    assert env["torch"] == "2.11.0+cu128" and "T4" in env.get("gpu", ""), env
    print("seed %d: %d epochs (history %d), %.2f h cumulative | AUC %.5f  rej@0.3 %.1f  final val loss %.5f"
          % (s, r["epochs_run"], len(h), r["walltime_sec"] / 3600, t["test_auc"], t["bg_rej_30"], vl[-1]))
    if r["epochs_run"] < EPOCHS:
        print("  INCOMPLETE (epoch %d of %d): finish before comparing" % (r["epochs_run"], EPOCHS))
        continue
    v60, a60, j60 = OWN60[s]
    print("  against its own epoch 60: dAUC %+.5f, d rej@0.3 %+.1f, d final val loss %+.5f"
          % (t["test_auc"] - a60, t["bg_rej_30"] - j60, vl[-1] - v60))
    print("  against the 120-epoch headline: dAUC %+.5f, d rej@0.3 %+.1f"
          % (t["test_auc"] - REF120[0], t["bg_rej_30"] - REF120[1]))
    if r.get("test_metrics_best_val"):
        tb = r["test_metrics_best_val"]
        print("  best-val weights (epoch %d): AUC %.5f, rej@0.3 %.1f"
              % (r["best_val_epoch"], tb["test_auc"], tb["bg_rej_30"]))
assert not BAD, "seeds failed (exit code or no result): %s" % BAD